# Semantic Kernel with Custom Models (Tencent Cloud/Deepseek)

In this code sample, you will use the [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI Framework with custom models like Tencent Cloud or Deepseek that support OpenAI-compatible APIs. 

The goal of this sample is to show you how to configure Semantic Kernel to work with your own model provider instead of GitHub Models.

## Import the Needed Python Packages 

In [ ]:
import os 
from typing import Annotated
from openai import AsyncOpenAI

from dotenv import load_dotenv

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function

# Import our custom model adapter
from model_adapter import get_semantic_kernel_config

## Creating the Client

In this sample, we will use a custom model API compatible with OpenAI format, such as Tencent Cloud or Deepseek. 

The `ai_model_id` is defined from your environment variables. You can change this to any model available on your provider.

For us to use your custom API, we will use the `OpenAIChatCompletion` connector within Semantic Kernel.

In [ ]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
# Load environment variables
load_dotenv()

# Get configuration from custom model adapter
sk_config = get_semantic_kernel_config()

# Verify configuration
api_key = sk_config['api_key']
endpoint = sk_config['endpoint']
model_id = sk_config['model_id']

if not all([api_key, endpoint, model_id]):
    print("❌ Error: Missing required environment variables!")
    print(f"  API Key: {'Set' if api_key else 'Missing'}")
    print(f"  Endpoint: {'Set' if endpoint else 'Missing'}")
    print(f"  Model ID: {'Set' if model_id else 'Missing'}")
else:
    print(f"✅ Using configuration:")
    print(f"  Endpoint: {endpoint}")
    print(f"  Model: {model_id}")
    print(f"  API Key: {'*' * (len(api_key) - 4) + api_key[-4:] if api_key else 'Not set'}")

# Create AsyncOpenAI client with custom endpoint
client = AsyncOpenAI(
    api_key=api_key, 
    base_url=endpoint,
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id=model_id,
    async_client=client,
)

## Creating the Agent 

Below we create the Agent called `TravelAgent`.

For this example, we are using very simple instructions. You can change these instructions to see how the agent responds differently. 

In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## Running the Agent

Now we can run the Agent by defining a thread of type `ChatHistoryAgentThread`.  Any required system messages are provided to the agent's invoke_stream `messages` keyword argument.

After these are defined, we create a `user_inputs` that will be what the user is sending to the agent. In this case, we have set this message to `Plan me a sunny vacation`. 

Feel free to change this message to see how the agent responds differently. 

In [ ]:
async def main():
    # Create a new thread for the agent
    # If no thread is provided, a new thread will be
    # created and returned with the initial response
    thread: ChatHistoryAgentThread | None = None

    user_inputs = [
        "Plan me a day trip.",
    ]

    for user_input in user_inputs:
        print(f"# User: {user_input}\n")
        first_chunk = True
        try:
            async for response in agent.invoke_stream(
                messages=user_input, thread=thread,
            ):
                # Print the response
                if first_chunk:
                    print(f"# {response.name}: ", end="", flush=True)
                    first_chunk = False
                print(f"{response}", end="", flush=True)
                thread = response.thread
            print()
        except Exception as e:
            print(f"❌ Error during agent execution: {e}")
            break

    # Clean up the thread
    await thread.delete() if thread else None

await main()